## Rasteryzing trees and Building heights 
Rasteryzing at 0.5m , aligned to the extention of the LST raster

### Libraries and general paths

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import os

In [ ]:

# Raster
lst_cropped_path = '../../../datos-notebooks/3.HR-data-bologna/3.0.base-data/LST/LST_cropped_celsius.tif' 

# Vectorials - inputs
buildings = "../../../datos-notebooks/3.HR-data-bologna/3.2.objects-height/bologne_buildings_height.json" # vectors of buildings with height column
trees = "../../../datos-notebooks/3.HR-data-bologna/3.2.objects-height/bologne_trees_height/" # folder with vector files of trees with height column

# Rasters - outputs
rasters_base = '../../../datos-notebooks/3.HR-data-bologna/3.2.objects-height/rasterized-0.5m/'
os.makedirs(rasters_base, exist_ok=True)
building_raster_path = os.path.join(rasters_base, 'buildings_height_0.5m.tif')
trees_raster_path = os.path.join(rasters_base, 'trees_height_0.5m.tif')

### Rasterizing biuldings

In [ ]:
# Load LST to get extent and CRS
with rasterio.open(lst_cropped_path) as src_lst:
    lst_crs = src_lst.crs
    bounds = src_lst.bounds  # xmin, ymin, xmax, ymax

# Load buildings
gdf_buildings = gpd.read_file(buildings)
gdf_buildings = gdf_buildings.to_crs(lst_crs)  # match LST CRS

# Define raster grid at 0.5 m
res = 0.5  # pixel size
width = int((bounds.right - bounds.left) / res)
height = int((bounds.top - bounds.bottom) / res)
transform = rasterio.transform.from_origin(bounds.left, bounds.top, res, res)

# Rasterize buildings using 'height' column
building_raster = rasterize(
    [(geom, value) for geom, value in zip(gdf_buildings.geometry, gdf_buildings.height)],
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    all_touched=True,
    dtype='float32'
)

# Save raster
with rasterio.open(
    building_raster_path,
    'w',
    driver='GTiff',
    height=building_raster.shape[0],
    width=building_raster.shape[1],
    count=1,
    dtype=building_raster.dtype,
    crs=lst_crs,
    transform=transform
) as dst:
    dst.write(building_raster, 1)

In [ ]:
# Print a window for checking the result
with rasterio.open(building_raster_path) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=7000, width=8000, height=8000)
    data = src.read(1, window=window)

plt.figure(figsize=(8,8))
plt.imshow(data, cmap='viridis')
plt.colorbar(label='Height (m)')
plt.show()

### Rasterizing trees

In [ ]:
# Load LST to get extent and CRS
with rasterio.open(lst_cropped_path) as src_lst:
    lst_crs = src_lst.crs
    bounds = src_lst.bounds  # xmin, ymin, xmax, ymax

# Define raster grid at 0.5 m
res = 0.5  # pixel size
width = int((bounds.right - bounds.left) / res)
height = int((bounds.top - bounds.bottom) / res)
transform = rasterio.transform.from_origin(bounds.left, bounds.top, res, res)

# Initialize empty raster 
tree_raster = np.full((height, width), np.nan, dtype='float32')

# Loop over all tree files in the folder
for file in os.listdir(trees):
    if file.endswith(".geojson") or file.endswith(".shp"):
        tree_path = os.path.join(trees, file)
        gdf_tree = gpd.read_file(tree_path)
        gdf_tree = gdf_tree.to_crs(lst_crs)

        # Rasterize this file
        temp_raster = rasterize(
            [(geom, value) for geom, value in zip(gdf_tree.geometry, gdf_tree.height)],
            out_shape=(height, width),
            transform=transform,
            fill=np.nan,
            all_touched=True,
            dtype='float32'
        )

        # Update only pixels that are still NaN
        mask = np.isnan(tree_raster) & ~np.isnan(temp_raster)
        tree_raster[mask] = temp_raster[mask]

# --- Save raster ---
with rasterio.open(
    trees_raster_path,
    'w',
    driver='GTiff',
    height=tree_raster.shape[0],
    width=tree_raster.shape[1],
    count=1,
    dtype=tree_raster.dtype,
    crs=lst_crs,
    transform=transform
) as dst:
    dst.write(tree_raster, 1)


In [ ]:
with rasterio.open(trees_raster_path) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=11000, width=8000, height=8000)  
    data = src.read(1, window=window)

plt.figure(figsize=(10,10))
plt.imshow(data, cmap='Greens')
plt.colorbar(label='Height (m)')
plt.show()